## Data Structure
A total of 50 questions are available.

They are divided into two blocks:

Block 1: Questions 1 to 25

Block 2: Questions 26 to 50

___

## Training Specification

The train_specification follows this format:

train_specification = 'starting question _ to _ end question _ percent'

    
Example:

    train_specification = '1_to_25_80p'

    
This means:

    Use questions 1 to 25 (Block 1)

    Use 80% of those questions and their evaluations for training 
    ~ 400 / 500 evaluations from Block 1

Corresponding file:
    
    Located in the training_splits folder => 'training_splits/train_1_to_25_80p.json'

## Test Specification

The test_specification uses the same format.

Example:

    test_specification = '26_to_50_100p'

This means:

    Use questions 26 to 50 (Block 2)

    Use 100% of those questions and their evaluations for testing 
    ~ 500 / 500 evaluations from Block 2
    
Corresponding file:

    Located in the test_splits folder => test_splits/test_26_to_50_100p.json

___

# Backupdata into training_split and test_split

In [1]:
import os

def extract_database_folder_names(folder):
    """ 
    Takes a folder name and extracts a list of shortened file names.
    
    Returns:
        A list of [str] where each str is a file name such as '_apixaban_2024-11-10_11-30.csv' is shortened to 'apixaban'.
    """
    
    def shorten_filename(filename):    
        parts = filename.split("_")
        if len(parts) > 2:
            shortened = "_".join(parts[:2])
        else:
            shortened = filename    
        return shortened[1:] if len(shortened) > 1 else ""

    return [
        shorten_filename(f)
        for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f))
    ]
    

database_folder_names = extract_database_folder_names("texts_v2")
database_folder_names

['ibuprofen adult',
 'treatment AND tuberculosis',
 'shock AND noradrenaline',
 'kawasaki AND treatment',
 'uncomplicated cystitis',
 'torsade de pointes',
 'aspirin kidney',
 'atorvastatin heart attack',
 'hepatitis AND treatment',
 'hiv test AND likelihood',
 'dual x-ray absorptiometry',
 'gastritis AND pregnancy',
 'malaria AND prevention',
 'myasthenia gravis',
 'gestational diabetes screening',
 'dabigatran',
 'lipase blood test',
 'aspirin heart attack',
 'fobt sensitivity',
 'pulmonary fibrosis',
 'parkinson AND dopamin',
 'clopidogrel AND stent',
 'apixaban',
 'cardiac catheterization',
 'colonoscopy']

In [2]:
import json

def _prepare_training_data():
    """
    Loads a JSON file containing annotated question-text pairs,
    filters out unwanted entries, and prepares a list of samples
    where both evaluators (Dennis and Karsten) agreed on the evaluation.

    Returns:
        A list of [question, text, evaluation] entries where both evaluators agreed.
    """

    # Load the JSON data from a local backup file
    with open('backups/backup_16_39_09_02_2025.json', 'r') as file:
        data = json.load(file)
        
    # Names of data sources to exclude entirely
    excludes = ['Dennis', 'Karsten', 'documents_pmc']

    # Only include data that is part of the second batch (filtered using folder names)
    include_data = database_folder_names
    data_queries = [x[1] for x in data if x[0] not in excludes and x[0] in include_data]

    # From each document collection, remove the 'text_id_referenz' entry and flatten the result
    all_docs = [
        docs[1] for documents in data_queries
        for docs in documents
        if docs[0] != 'text_id_referenz'
    ]
    
    trainings_list = []
    for doc in all_docs:
        dennis_evaluation = doc['evaluation']['Dennis']
        karsten_evaluation = doc['evaluation']['Karsten']
        text_of_document = doc['Text']
        question_of_document = doc['Question']
        
        # Only include documents where both evaluators agreed on the evaluation
        if dennis_evaluation == karsten_evaluation:
            trainings_list.append([
                question_of_document,
                text_of_document,
                karsten_evaluation
            ])
            
    return trainings_list

data = _prepare_training_data()
len(data)

327

In [3]:
import json
from sklearn.model_selection import train_test_split

def data_split_as_json():
    """takes the split data and saves them to seperate files"""
    
    # Extract questions, answers, and labels from the data
    queries = [item[0] for item in data]   # List of all questions
    answers = [item[1] for item in data]   # List of corresponding texts
    labels = [item[2] for item in data]    # List of binary labels (0 or 1)

    # Split the data into training and test sets (80% training, 20% test)
    query_train, query_test, answer_train, answer_test, label_train, label_test = train_test_split(
        queries, answers, labels, test_size=0.2, random_state=42
    )
    # Recombine the splits into structured (question, answer, label) tuples
    train_data = list(zip(query_train, answer_train, label_train))
    test_data = list(zip(query_test, answer_test, label_test))

    # train_26_to_50_80p means => Trainingdata from Question 26 to 50 and 80p(percent) of the data
    filename_train = f'training_splits/train_26_to_50_80p.json'    
    with open(filename_train, 'w') as json_file:
        json.dump(train_data, json_file, indent=4)
    
    # test_26_to_50_80p means => Testdata from Question 26 to 50 and 20p(percent) of the data
    filename_test = f'test_splits/test_26_to_50_20p.json'
    with open(filename_test, 'w') as json_file:
        json.dump(test_data, json_file, indent=4)
        
data_split_as_json()